In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.optim as optim

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
dataLoader = DataLoader(datasets.MNIST(root='./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)

In [ ]:
from torch.nn.modules import ReLU
class NNModel(nn.Module):
  def __init__(self,inputSize=28*28):

    super().__init__()

    self.model = nn.Sequential(

      nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),

      nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),

      nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),

      nn.Conv2d(128, 64, kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),

      nn.Flatten(),

      nn.Linear(64, 32),
      nn.ReLU(),

      nn.Linear(32, 10),
      # nn.Softmax()

    )
  def forward(self,imgSize):
    return self.model(imgSize)

In [ ]:
nnModel = NNModel().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.Adam(nnModel.parameters(), lr=0.0003)

In [ ]:
for epoch in range(10):
  for img,labels in dataLoader:

    img = img.to(device)
    batchSize = img.size(0)

    labels = labels.to(device)

    loss = criterion(nnModel(img), labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f"epoch: {epoch}, Loss: {loss.item():.4f}")

epoch: 0, Loss: 0.2358
epoch: 1, Loss: 0.0448
epoch: 2, Loss: 0.1528
epoch: 3, Loss: 0.0693
epoch: 4, Loss: 0.0168
epoch: 5, Loss: 0.0047
epoch: 6, Loss: 0.0499
epoch: 7, Loss: 0.0070
epoch: 8, Loss: 0.0011
epoch: 9, Loss: 0.0392
epoch: 10, Loss: 0.0098
epoch: 11, Loss: 0.0210
epoch: 12, Loss: 0.0027
epoch: 13, Loss: 0.0001
epoch: 14, Loss: 0.0001
epoch: 15, Loss: 0.0019
epoch: 16, Loss: 0.0071
epoch: 17, Loss: 0.0110
epoch: 18, Loss: 0.0011
epoch: 19, Loss: 0.0086


In [ ]:
test_loader = DataLoader(
    datasets.MNIST(root='./data', train=False, download=True, transform=transform),
    batch_size=64,
    shuffle=False
)

In [ ]:
nnModel.eval()

test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for img, labels in test_loader:
        img, labels = img.to(device), labels.to(device)

        outputs = nnModel(img)
        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_loss = test_loss / len(test_loader)
accuracy = correct / total

print(f"Test Loss: {avg_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.0223
Test Accuracy: 0.9928
